# Phase 9: Results and Visualization
## Project: Workload-Aware SARIMA-GARCH Framework for Data Center Cooling Load Forecasting and Uncertainty Analysis

This notebook generates publication-quality figures presenting empirical results across point forecasting accuracy, conditional volatility tracking, dynamic prediction intervals, workload regime calibration, residual diagnostics, and the coverage vs. sharpness Pareto frontier.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from arch import arch_model
from statsmodels.graphics.tsaplots import plot_acf

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['figure.titlesize'] = 14

out_dir = '../results/figures'
os.makedirs(out_dir, exist_ok=True)

df_clean = pd.read_csv('../data/processed/cleaned_data.csv', parse_dates=['timestamp'])
df_test = pd.read_csv('../results/forecasts/forecast_results.csv', parse_dates=['timestamp'])
model_metrics = pd.read_csv('../results/metrics/model_metrics.csv')

print(f"Loaded test results: {len(df_test)} hours")
display(model_metrics)

### 1. Multi-Week Actual vs. Predicted Cooling Power
Evaluating point forecast tracking and residual tracking on the holdout test set.

In [2]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [2.5, 1]})
sample_df = df_test.iloc[100:604].copy()

ax1.plot(sample_df['timestamp'], sample_df['actual'], label='Actual Cooling Power (kW)', color='#1f77b4', linewidth=1.8, alpha=0.9)
ax1.plot(sample_df['timestamp'], sample_df['calib_u95'] - (sample_df['calib_u95'] - sample_df['calib_l95'])/2, 
         label='Proposed Calibrated SARIMAX Forecast', color='#d62728', linestyle='--', linewidth=1.5, alpha=0.9)
ax1.fill_between(sample_df['timestamp'], sample_df['calib_l95'], sample_df['calib_u95'], 
                 color='#ff9896', alpha=0.35, label='Calibrated 95% Dynamic Prediction Interval')
ax1.set_ylabel('Cooling Power (kW)', fontweight='bold')
ax1.set_title('Figure 1: Multi-Week Actual vs. Calibrated SARIMAX-GARCH Cooling Power Forecast (Holdout Test Set)', fontweight='bold', pad=12)
ax1.legend(loc='upper right', frameon=True, framealpha=0.9)
ax1.grid(True, linestyle=':', alpha=0.6)

resids = sample_df['actual'] - sample_df['sarimax_pred']
ax2.plot(sample_df['timestamp'], resids, color='#2ca02c', linewidth=1.2, label='Point Forecast Residual ($e_t = y_t - \\hat{y}_t$)')
ax2.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax2.fill_between(sample_df['timestamp'], 0, resids, color='#98df8a', alpha=0.3)
ax2.set_ylabel('Residual (kW)', fontweight='bold')
ax2.set_xlabel('Timestamp', fontweight='bold')
ax2.legend(loc='upper right', frameon=True, framealpha=0.9)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
plt.tight_layout()
plt.show()

### 2. Multi-Model Forecast Comparison
Comparing Seasonal-Naive, SARIMA baseline, and Proposed Workload-Aware SARIMAX.

In [3]:
fig, ax = plt.subplots(figsize=(14, 6))
zoom_df = df_test.iloc[200:440].copy()

ax.plot(zoom_df['timestamp'], zoom_df['actual'], label='Actual Realization', color='#111111', linewidth=2.2, zorder=5)
ax.plot(zoom_df['timestamp'], zoom_df['naive_pred'], label='Seasonal-Naive ($y_{t-24}$)', color='#7f7f7f', linestyle=':', linewidth=1.5, alpha=0.75)
ax.plot(zoom_df['timestamp'], zoom_df['sarima_pred'], label='SARIMA(1,0,1)(1,1,1)24 Baseline', color='#1f77b4', linestyle='-.', linewidth=1.6, alpha=0.85)
ax.plot(zoom_df['timestamp'], zoom_df['sarimax_pred'], label='Workload-Aware SARIMAX (Proposed Point)', color='#d62728', linestyle='-', linewidth=1.8, alpha=0.9)
ax.set_title('Figure 2: Multi-Model Point Forecast Tracking Across Dynamic IT Workload Cycles', fontweight='bold', pad=12)
ax.set_xlabel('Timestamp (Hourly Resolution)', fontweight='bold')
ax.set_ylabel('Cooling Power (kW)', fontweight='bold')
ax.legend(loc='upper right', frameon=True, framealpha=0.9)
ax.grid(True, linestyle=':', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
plt.tight_layout()
plt.show()

### 3. Annual Conditional Volatility and Weather Dynamics

In [4]:
train_n = 6132
y_series = df_clean['cooling_kw']
diff_y = y_series.diff(24).dropna()
full_garch_res = arch_model(diff_y, vol='Garch', p=1, q=1, dist='StudentsT', mean='Zero').fit(disp='off')
vol_series = full_garch_res.conditional_volatility

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1.2]})
timestamps_vol = df_clean['timestamp'].iloc[24:]
ax1.plot(timestamps_vol, vol_series, color='#9467bd', linewidth=1.2, label=r'GARCH(1,1)-$t$ Conditional Volatility $\hat{\sigma}_t$ (kW)')
ax1.axvline(df_clean['timestamp'].iloc[train_n], color='black', linestyle='--', linewidth=1.2, label='Train / Validation Split')
ax1.axvline(df_clean['timestamp'].iloc[train_n + 1314], color='blue', linestyle='--', linewidth=1.2, label='Validation / Test Split')
ax1.set_ylabel(r'Volatility $\hat{\sigma}_t$ (kW)', fontweight='bold')
ax1.set_title(r'Figure 3: Annual Time-Varying Conditional Volatility $\hat{\sigma}_t$ and Ambient Temperature Interaction', fontweight='bold', pad=12)
ax1.legend(loc='upper right', frameon=True, framealpha=0.9)
ax1.grid(True, linestyle=':', alpha=0.6)

ax2.plot(timestamps_vol, df_clean['outdoor_air_temp'].iloc[24:], color='#ff7f0e', linewidth=1.1, label='Outdoor Temperature (°F)')
ax2.axvline(df_clean['timestamp'].iloc[train_n], color='black', linestyle='--', linewidth=1.2)
ax2.axvline(df_clean['timestamp'].iloc[train_n + 1314], color='blue', linestyle='--', linewidth=1.2)
ax2.set_ylabel('Outdoor Temp (°F)', fontweight='bold')
ax2.set_xlabel('Timestamp (May 2024 – Apr 2025)', fontweight='bold')
ax2.legend(loc='upper right', frameon=True, framealpha=0.9)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

### 4. Dynamic Prediction Intervals (80% and 95%)

In [5]:
slice_df = df_test.iloc[300:636].copy()

# 80% interval comparison
fig, ax = plt.subplots(figsize=(14, 5))
fixed_half_80 = 6.105 / 2.0
ax.plot(slice_df['timestamp'], slice_df['actual'], color='black', linewidth=1.8, label='Actual Cooling Power (kW)', zorder=6)
ax.fill_between(slice_df['timestamp'], slice_df['sarima_pred'] - fixed_half_80, slice_df['sarima_pred'] + fixed_half_80, 
                color='#1f77b4', alpha=0.15, label='Fixed SARIMA 80% Band (MPIW=6.11 kW)')
ax.fill_between(slice_df['timestamp'], slice_df['calib_l80'], slice_df['calib_u80'], 
                color='#d62728', alpha=0.3, label='Proposed Calibrated 80% Band (MPIW=2.05 kW)')
ax.set_title('Figure 4: Dynamic 80% Prediction Interval Comparison', fontweight='bold', pad=12)
ax.set_ylabel('Cooling Power (kW)', fontweight='bold')
ax.legend(loc='upper right', frameon=True)
ax.grid(True, linestyle=':', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
plt.tight_layout()
plt.show()

# 95% interval comparison
fig, ax = plt.subplots(figsize=(14, 5))
fixed_half_95 = 11.188 / 2.0
ax.plot(slice_df['timestamp'], slice_df['actual'], color='black', linewidth=1.8, label='Actual Cooling Power (kW)', zorder=6)
ax.fill_between(slice_df['timestamp'], slice_df['sarima_pred'] - fixed_half_95, slice_df['sarima_pred'] + fixed_half_95, 
                color='#1f77b4', alpha=0.15, label='Fixed SARIMA 95% Band (MPIW=11.19 kW)')
ax.fill_between(slice_df['timestamp'], slice_df['calib_l95'], slice_df['calib_u95'], 
                color='#2ca02c', alpha=0.3, label='Proposed Calibrated 95% Band (MPIW=4.13 kW, PICP=93.61% / 96.40% under High Workload)')
ax.set_title('Figure 5: Dynamic 95% Prediction Interval Comparison', fontweight='bold', pad=12)
ax.set_ylabel('Cooling Power (kW)', fontweight='bold')
ax.legend(loc='upper right', frameon=True)
ax.grid(True, linestyle=':', alpha=0.6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
plt.tight_layout()
plt.show()

### 5. Workload Regime Breakdown and Residual Diagnostics

In [6]:
# Pareto frontier
fig, ax = plt.subplots(figsize=(10, 6))
models = [
    ('Naive 80%', 21.404, 96.19, '#7f7f7f', 's'),
    ('Naive 95%', 32.734, 98.48, '#7f7f7f', 's'),
    ('SARIMA 80%', 6.105, 96.35, '#1f77b4', '^'),
    ('SARIMA 95%', 11.188, 99.24, '#1f77b4', '^'),
    ('SARIMA-GARCH 80%', 1.965, 74.28, '#ff7f0e', 'o'),
    ('SARIMA-GARCH 95%', 3.600, 91.25, '#ff7f0e', 'o'),
    ('Proposed SARIMAX-GARCH 80%', 2.051, 75.57, '#d62728', '*'),
    ('Proposed SARIMAX-GARCH 95%', 4.129, 93.61, '#d62728', '*'),
]
for label, mpiw, picp, color, marker in models:
    size = 180 if marker == '*' else 100
    ax.scatter(mpiw, picp, color=color, marker=marker, s=size, edgecolors='black', linewidth=1, zorder=5)
    xytext = (10, -5) if 'Proposed' in label or 'Naive' in label else (-10, 8)
    ax.annotate(label, (mpiw, picp), textcoords="offset points", xytext=xytext,
                fontweight='bold' if 'Proposed' in label else 'normal',
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.85, lw=1))
ax.axhline(80.0, color='gray', linestyle=':', linewidth=1.2, label='Nominal 80% Level')
ax.axhline(95.0, color='red', linestyle='--', linewidth=1.2, label='Nominal 95% Level')
ax.set_title('Figure 8: Empirical Uncertainty Pareto Trade-Off: PICP vs. MPIW', fontweight='bold', pad=12)
ax.set_xlabel('Mean Prediction Interval Width - MPIW (kW)', fontweight='bold')
ax.set_ylabel('Coverage Probability - PICP (%)', fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()